# Algoritmo Genético para o CVRP

**Disciplina:** Inteligência Artificial e Aprendizado de Máquina — 2026/1  
**Professor:** Gabriel de Oliveira Ramos  
**Grupo 8:** Erik Morbach, Gabriel Farias, Lucas Escopelli

---

## Resumo

Este trabalho apresenta a implementação de um Algoritmo Genético (AG) para o *Capacitated Vehicle Routing Problem* (CVRP). O CVRP é um problema NP-difícil: dado um depósito e um conjunto de cidades com demandas individuais, deve-se determinar rotas para uma frota de veículos com capacidade limitada, minimizando (1) o número de veículos e (2) a distância total percorrida.

Implementamos e comparamos **sete operadores genéticos**: dois de seleção (Torneio k=3 e k=7), dois de cruzamento (OX e PMX) e três de mutação (Swap, 2-opt e Or-opt). A população é inicializada com uma heurística de pivôs geográficos, e avaliamos também o impacto dessa heurística em relação à inicialização puramente aleatória. Os experimentos são executados nas três instâncias fornecidas: `eil33`, `eil51` e `eil76`.

**Link do vídeo:** *(a preencher antes da entrega)*

## 1. Descrição do Problema

O **Capacitated Vehicle Routing Problem (CVRP)** consiste em, dado:
- Um **depósito** central (ponto de partida e chegada de todos os veículos),
- Um conjunto de **cidades** com coordenadas $(x, y)$ e demanda $d_i$,
- Uma **capacidade máxima** $Q$ para cada veículo,

encontrar um conjunto de rotas tal que:
- Cada cidade seja visitada **exatamente uma vez**;
- A soma das demandas de cada rota **não ultrapasse** $Q$;
- O **número de veículos** seja mínimo;
- A **distância total** percorrida seja mínima (objetivo secundário).

O problema é **NP-difícil**, o que torna inviável a busca exaustiva para instâncias com mais de ~20 cidades. Isso justifica o uso de meta-heurísticas como Algoritmos Genéticos.

## 2. Leitura e Visualização dos Dados

As instâncias estão no formato `.vrp` (TSPLIB). A função abaixo lê o arquivo e retorna as coordenadas, demandas, capacidade e o nó depósito.

In [ ]:
import math
import random
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# ── Caminho base das instâncias ──────────────────────────────────────────
# Ajuste este caminho conforme seu ambiente (local ou Google Colab)
INSTANCES_DIR = "/home/lucas/Faculdade/AI/trabGA/instances/CVRP/"

def load_vrp(filepath):
    """Lê um arquivo .vrp e retorna (nodes, node_cap, cap, depot)."""
    nodes = {}; node_cap = {}; cap = 0; depot = None; state = 0
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line == 'EOF': continue
            if 'CAPACITY' in line and ':' in line:
                cap = int(line.split(':')[1])
            elif 'NODE_COORD_SECTION' in line: state = 1
            elif 'DEMAND_SECTION'    in line: state = 2
            elif 'DEPOT_SECTION'     in line: state = 3
            elif state == 1:
                parts = line.split()
                if len(parts) >= 3:
                    nid, x, y = int(parts[0]), int(parts[1]), int(parts[2])
                    nodes[nid] = (x, y)
            elif state == 2:
                parts = line.split()
                if len(parts) >= 2:
                    nid, demand = int(parts[0]), int(parts[1])
                    node_cap[nid] = demand
            elif state == 3:
                try:
                    nid = int(line)
                    if nid != -1 and depot is None: depot = nid
                except ValueError: pass
    if depot is None:
        depot = next(n for n, d in node_cap.items() if d == 0)
    return nodes, node_cap, cap, depot


def dist(a, b, nodes):
    """Distância euclidiana entre dois nós."""
    dx = nodes[a][0] - nodes[b][0]; dy = nodes[a][1] - nodes[b][1]
    return math.sqrt(dx * dx + dy * dy)


def visualize_instance(nodes, node_cap, depot, title="Instância"):
    """Plota as cidades com suas demandas."""
    fig, ax = plt.subplots(figsize=(8, 6))
    for nid, (x, y) in nodes.items():
        if nid == depot:
            ax.scatter(x, y, s=200, color='black', zorder=5, marker='s')
            ax.annotate('Depósito', (x, y), textcoords='offset points', xytext=(5, 5))
        else:
            ax.scatter(x, y, s=60, color='steelblue', zorder=4)
            ax.annotate(str(node_cap[nid]), (x, y), textcoords='offset points',
                        xytext=(3, 3), fontsize=7)
    ax.set_title(title); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


# Carregar e visualizar eil33
nodes, node_cap, cap, depot = load_vrp(INSTANCES_DIR + "eil33.vrp")
print(f"Instância eil33: {len(nodes)} nós, capacidade={cap}, depósito={depot}")
print(f"Demanda total: {sum(node_cap.values())},  "
      f"veículos mínimos: {math.ceil(sum(node_cap.values()) / cap)}")
visualize_instance(nodes, node_cap, depot, title="eil33 — cidades e demandas")

## 3. Representação dos Indivíduos

Cada **indivíduo** (solução candidata) é representado como uma **permutação plana** das cidades (excluindo o depósito):

```
[c3, c7, c12, c2, c15, c8, c5, c11, ...]
```

A divisão em rotas **não está codificada no indivíduo** — ela é calculada pela função de avaliação usando um algoritmo *greedy*: percorre a permutação da esquerda para a direita e abre uma nova rota sempre que adicionar a próxima cidade excederia a capacidade do veículo.

**Vantagem desta representação:** os operadores de cruzamento clássicos (OX, PMX) foram desenvolvidos para permutações e funcionam naturalmente aqui, sem precisar tratar rotas explicitamente.

## 4. Função de Avaliação (Fitness)

A avaliação de um indivíduo é feita em duas etapas:

1. **Decodificação greedy:** percorre a permutação e abre uma nova rota sempre que a próxima cidade excederia a capacidade. Cada rota começa e termina no depósito.
2. **Cálculo do custo:** retorna `(n_veículos, distância_total)`. Por comparação lexicográfica de tuplas, o GA minimiza veículos primeiro e distância depois.

```
Permutação:  [c3, c7, c12, c2, c15, ...]    capacidade = 100

c3  (dem 30) → rota atual: [dep, c3],       carga: 30
c7  (dem 40) → rota atual: [dep, c3, c7],   carga: 70
c12 (dem 40) → 70+40=110 > 100  → fecha rota 1: [dep,c3,c7,dep]
             → abre rota 2: [dep, c12],     carga: 40
...
```

In [ ]:
def decode_routes(perm, node_cap, cap, depot):
    """Converte uma permutação de cidades em rotas viáveis (greedy)."""
    routes = []; current_route = [depot]; current_load = 0
    for city in perm:
        demand = node_cap[city]
        if current_load + demand <= cap:
            current_route.append(city); current_load += demand
        else:
            current_route.append(depot); routes.append(current_route)
            current_route = [depot, city]; current_load = demand
    current_route.append(depot); routes.append(current_route)
    return routes


def route_distance(route, nodes):
    """Distância total de uma rota (inclui ida e volta ao depósito)."""
    return sum(dist(route[i], route[i + 1], nodes) for i in range(len(route) - 1))


def evaluate(perm, nodes, node_cap, cap, depot):
    """
    Retorna (n_veículos, distância_total).
    Tupla: minimiza veículos primeiro, depois distância.
    """
    routes = decode_routes(perm, node_cap, cap, depot)
    return (len(routes), sum(route_distance(r, nodes) for r in routes))


def plot_solution(perm, nodes, node_cap, cap, depot, title="Solução"):
    """Plota as rotas de uma solução."""
    routes = decode_routes(perm, node_cap, cap, depot)
    colors = [cm.tab10(i / max(len(routes), 1)) for i in range(len(routes))]
    fig, ax = plt.subplots(figsize=(9, 7))
    for i, route in enumerate(routes):
        xs = [nodes[c][0] for c in route]; ys = [nodes[c][1] for c in route]
        ax.plot(xs, ys, '-o', color=colors[i], alpha=0.8,
                linewidth=1.5, markersize=5, label=f'Rota {i+1}')
    ax.scatter(*nodes[depot], s=250, color='black', zorder=6, marker='s', label='Depósito')
    n_trucks, total_dist = evaluate(perm, nodes, node_cap, cap, depot)
    ax.set_title(f"{title}\n{n_trucks} veículos, distância total = {total_dist:.1f}")
    ax.legend(loc='upper right', fontsize=7); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

## 5. Inicialização da População

A função `initialize_population` aceita a flag `use_pivot`:

- `use_pivot=True` *(padrão)*: gera 1 indivíduo via heurística de pivôs + `pop_size-1` aleatórios.
- `use_pivot=False`: gera toda a população com permutações aleatórias.

O **Experimento 5** compara diretamente as duas estratégias.

### Heurística de Pivôs
1. Calcula $k = \lceil \text{demanda total} / Q \rceil$ (veículos mínimos).
2. Seleciona $k$ **pivôs** espalhados geograficamente (*farthest-first*): o primeiro é o mais distante do depósito; cada próximo maximiza a distância mínima aos pivôs já escolhidos.
3. Associa cada cidade ao pivô mais próximo.
4. Ordena cidades dentro de cada grupo por *nearest neighbor* a partir do pivô.
5. Concatena os grupos para formar a permutação.

In [ ]:
def select_pivots(n_pivots, cities, nodes, depot):
    """Seleciona n_pivots cidades usando farthest-first."""
    pivots = [max(cities, key=lambda c: dist(c, depot, nodes))]
    while len(pivots) < n_pivots:
        remaining = [c for c in cities if c not in pivots]
        nxt = max(remaining, key=lambda c: min(dist(c, p, nodes) for p in pivots))
        pivots.append(nxt)
    return pivots


def nearest_neighbor_order(start, group, nodes):
    """Ordena 'group' por nearest neighbor a partir de 'start'."""
    remaining = [c for c in group if c != start]
    order = [start]; current = start
    while remaining:
        nearest = min(remaining, key=lambda c: dist(current, c, nodes))
        order.append(nearest); remaining.remove(nearest); current = nearest
    return order


def pivot_individual(cities, nodes, node_cap, cap, depot):
    """Gera um indivíduo usando a heurística de pivôs."""
    n_trucks = math.ceil(sum(node_cap[c] for c in cities) / cap)
    pivots   = select_pivots(n_trucks, cities, nodes, depot)
    groups   = {p: [] for p in pivots}
    for city in cities:
        if city in pivots: continue
        nearest = min(pivots, key=lambda p: dist(city, p, nodes))
        groups[nearest].append(city)
    perm = []
    for pivot in pivots:
        perm.extend(nearest_neighbor_order(pivot, groups[pivot], nodes))
    return perm


def initialize_population(pop_size, cities, nodes, node_cap, cap, depot,
                           seed=None, use_pivot=True):
    """
    Cria a população inicial.

    use_pivot=True  → 1 indivíduo via heurística de pivôs + resto aleatório
    use_pivot=False → toda a população é aleatória
    """
    if seed is not None:
        random.seed(seed)
    pop = [pivot_individual(cities, nodes, node_cap, cap, depot)] if use_pivot else []
    while len(pop) < pop_size:
        p = cities[:]; random.shuffle(p); pop.append(p)
    return pop


# Comparação rápida: pivô vs. aleatório
cities_33 = [n for n in nodes if n != depot]
pop_test  = initialize_population(5, cities_33, nodes, node_cap, cap, depot, seed=0)
fit_pivot  = evaluate(pop_test[0], nodes, node_cap, cap, depot)
fit_random = evaluate(pop_test[1], nodes, node_cap, cap, depot)
print(f"Pivô:     {fit_pivot[0]} veículos, distância = {fit_pivot[1]:.1f}")
print(f"Aleatório:{fit_random[0]} veículos, distância = {fit_random[1]:.1f}")
plot_solution(pop_test[0], nodes, node_cap, cap, depot, "Solução Inicial — Heurística de Pivôs (eil33)")

## 6. Operadores Genéticos

Implementamos sete operadores, distribuídos em três categorias conforme exigido pelo enunciado.

### 6.1 Seleção

**Torneio de tamanho k:** sorteia `k` indivíduos aleatoriamente e retorna o de melhor fitness.

| Operador | k | Característica |
|---|---|---|
| `tournament_k3` | 3 | Pressão seletiva moderada — mais diversidade |
| `tournament_k7` | 7 | Pressão seletiva alta — converge mais rápido |

In [ ]:
def tournament_selection(population, fitnesses, k):
    indices = random.sample(range(len(population)), k)
    best = min(indices, key=lambda i: fitnesses[i])
    return population[best][:]

def tournament_k3(population, fitnesses):
    """Seleção por torneio com k=3."""
    return tournament_selection(population, fitnesses, 3)

def tournament_k7(population, fitnesses):
    """Seleção por torneio com k=7."""
    return tournament_selection(population, fitnesses, 7)

### 6.2 Cruzamento (Crossover)

Ambos os operadores garantem que cada cidade apareça **exatamente uma vez** no filho.

**OX — Order Crossover:**
1. Copia um segmento aleatório do Pai 1 para o filho.
2. Preenche o restante com as cidades do Pai 2 na ordem em que aparecem, pulando as já copiadas.

**PMX — Partially Mapped Crossover:**
1. Copia um segmento do Pai 1.
2. Usa um mapeamento entre os pais para resolver conflitos de posição.
3. Preenche o restante com o Pai 2.

O PMX preserva melhor as posições absolutas; o OX preserva melhor a ordem relativa.

In [ ]:
def ox_crossover(p1, p2):
    """Order Crossover (OX)."""
    n = len(p1); a, b = sorted(random.sample(range(n), 2))
    child = [None] * n; child[a:b+1] = p1[a:b+1]
    seg = set(p1[a:b+1]); rem = [x for x in p2 if x not in seg]
    idx = 0
    for i in list(range(b+1, n)) + list(range(0, a)):
        child[i] = rem[idx]; idx += 1
    return child


def pmx_crossover(p1, p2):
    """Partially Mapped Crossover (PMX)."""
    n = len(p1); a, b = sorted(random.sample(range(n), 2))
    child = [None] * n; child[a:b+1] = p1[a:b+1]
    seg = set(p1[a:b+1]); pos_p2 = {v: i for i, v in enumerate(p2)}
    for i in range(a, b+1):
        val = p2[i]
        if val not in seg:
            pos = i
            while child[pos] is not None:
                pos = pos_p2[child[pos]]
            child[pos] = val
    for i in range(n):
        if child[i] is None: child[i] = p2[i]
    return child

### 6.3 Mutação

| Operador | Descrição | Intensidade |
|---|---|---|
| **Swap** | Troca duas cidades de posição | Baixa |
| **2-opt** | Inverte um segmento da permutação | Média |
| **Or-opt** | Move um bloco de 2–3 cidades para outra posição | Alta |

**Or-opt** é o **7º operador** exigido pelo enunciado. É amplamente utilizado na literatura de VRP e mais poderoso que o Swap porque move blocos de cidades consecutivas, preservando sub-sequências que já são eficientes.

In [ ]:
def swap_mutation(perm):
    """Troca duas cidades de posição aleatoriamente."""
    c = perm[:]; i, j = random.sample(range(len(perm)), 2)
    c[i], c[j] = c[j], c[i]; return c


def two_opt_mutation(perm):
    """Inverte um segmento aleatório da permutação (2-opt)."""
    c = perm[:]; i, j = sorted(random.sample(range(len(perm)), 2))
    c[i:j+1] = c[i:j+1][::-1]; return c


def or_opt_mutation(perm):
    """
    Move um bloco de 2 ou 3 cidades consecutivas para outra posição (Or-opt).
    Preserva sub-sequências localmente boas — mais poderoso que o Swap.
    """
    c = perm[:]; n = len(c)
    bs = random.choice([2, 3])
    if n <= bs + 1: return swap_mutation(perm)   # fallback para permutações curtas
    i     = random.randint(0, n - bs)
    block = c[i:i+bs]; rest = c[:i] + c[i+bs:]
    j     = random.randint(0, len(rest))
    return rest[:j] + block + rest[j:]

## 7. Loop Principal do Algoritmo Genético

O loop evolutivo usa **elitismo**: o melhor indivíduo de cada geração é sempre preservado.

A flag `use_pivot` é propagada até a inicialização da população, permitindo comparar as duas estratégias nos experimentos.

```
Para cada geração:
  1. Preserva o melhor indivíduo (elitismo)
  2. Repete até completar a nova população:
     a. Seleciona dois pais
     b. Aplica cruzamento (prob. p_cross)
     c. Aplica mutação   (prob. p_mut)
  3. Avalia todos os filhos
  4. Substitui a população
```

In [ ]:
def run_ga(nodes, node_cap, cap, depot,
           pop_size=80, n_gen=300,
           p_cross=0.85, p_mut=0.15,
           selection_fn=tournament_k3,
           crossover_fn=ox_crossover,
           mutation_fn=swap_mutation,
           seed=42, use_pivot=True, verbose=False):
    """
    Executa o Algoritmo Genético para o CVRP.

    Parâmetros:
      use_pivot : True  → 1 indivíduo inicial via heurística de pivôs
                  False → população inicial totalmente aleatória

    Retorna dict com:
      best_individual   : melhor permutação encontrada
      best_fitness      : (n_veículos, distância) do melhor
      best_dist_history : distância do melhor por geração
      avg_dist_history  : distância média da população por geração
    """
    random.seed(seed)
    cities = [n for n in nodes if n != depot]

    population = initialize_population(pop_size, cities, nodes, node_cap, cap, depot,
                                        seed=seed, use_pivot=use_pivot)
    fitnesses = [evaluate(ind, nodes, node_cap, cap, depot) for ind in population]

    best_dist_history = []
    avg_dist_history  = []

    for gen in range(n_gen):
        new_pop = []

        # Elitismo: preserva o melhor indivíduo
        best_idx = min(range(len(population)), key=lambda i: fitnesses[i])
        new_pop.append(population[best_idx][:])

        while len(new_pop) < pop_size:
            p1 = selection_fn(population, fitnesses)
            p2 = selection_fn(population, fitnesses)
            child = crossover_fn(p1, p2) if random.random() < p_cross else p1[:]
            child = mutation_fn(child)   if random.random() < p_mut   else child
            new_pop.append(child)

        population = new_pop
        fitnesses  = [evaluate(ind, nodes, node_cap, cap, depot) for ind in population]

        best_fit = min(fitnesses)
        best_dist_history.append(best_fit[1])
        avg_dist_history.append(sum(f[1] for f in fitnesses) / len(fitnesses))

        if verbose and (gen + 1) % 50 == 0:
            print(f"  Geração {gen+1:4d}: {best_fit[0]} veículos, dist={best_fit[1]:.1f}")

    best_idx = min(range(len(population)), key=lambda i: fitnesses[i])
    return {
        'best_individual'  : population[best_idx],
        'best_fitness'     : fitnesses[best_idx],
        'best_dist_history': best_dist_history,
        'avg_dist_history' : avg_dist_history,
    }

## 8. Experimentos

### Metodologia

Os experimentos são organizados em cinco blocos:

| Bloco | O que varia | Fixo | Instância |
|---|---|---|---|
| 1 — Seleção | Torneio k=3 vs k=7 | OX, Swap, pop=80, gen=300 | eil33 |
| 2 — Cruzamento | OX vs PMX | Torneio k=3, Swap, pop=80, gen=300 | eil33 |
| 3 — Mutação | Swap vs 2-opt vs Or-opt | Torneio k=3, OX, pop=80, gen=300 | eil33 |
| 4 — Melhor config. | — | melhor dos blocos 1–3 | eil33, eil51, eil76 |
| 5 — Inicialização | `use_pivot=True` vs `False` | melhor config. | eil33, eil51, eil76 |

Cada configuração é executada com **3 sementes** e os resultados são a média entre as execuções.

In [ ]:
# Carrega as três instâncias
instances = {}
for name in ['eil33', 'eil51', 'eil76']:
    n, nc, c, d = load_vrp(INSTANCES_DIR + f"{name}.vrp")
    instances[name] = {'nodes': n, 'node_cap': nc, 'cap': c, 'depot': d,
                       'cities': [x for x in n if x != d]}
    total = sum(nc.values())
    print(f"{name}: {len(n)} nós, cap={c}, veículos mínimos={math.ceil(total/c)}")

In [ ]:
def run_experiment(inst_name, configs, pop_size=80, n_gen=300,
                   p_cross=0.85, p_mut=0.15,
                   seeds=(0, 1, 2), use_pivot=True):
    """
    Executa múltiplas configurações numa instância com várias sementes.
    Retorna: config_name → {best_dist_history, avg_dist_history, best_fitness}
    """
    inst = instances[inst_name]
    results = {}
    for cfg_name, sel_fn, cross_fn, mut_fn in configs:
        print(f"  [{inst_name}] {cfg_name} ...", end=' ', flush=True)
        all_best = []; all_avg = []; all_fit = []
        for seed in seeds:
            res = run_ga(inst['nodes'], inst['node_cap'], inst['cap'], inst['depot'],
                         pop_size=pop_size, n_gen=n_gen,
                         p_cross=p_cross, p_mut=p_mut,
                         selection_fn=sel_fn, crossover_fn=cross_fn, mutation_fn=mut_fn,
                         seed=seed, use_pivot=use_pivot)
            all_best.append(res['best_dist_history'])
            all_avg.append(res['avg_dist_history'])
            all_fit.append(res['best_fitness'])
        results[cfg_name] = {
            'best_dist_history': [sum(h[g] for h in all_best)/len(seeds) for g in range(n_gen)],
            'avg_dist_history' : [sum(h[g] for h in all_avg) /len(seeds) for g in range(n_gen)],
            'best_fitness'     : min(all_fit),
        }
        bf = results[cfg_name]['best_fitness']
        print(f"veículos={bf[0]}, dist={bf[1]:.1f}")
    return results


def plot_experiment(results, title, ylabel="Distância"):
    """Plota curvas de evolução (melhor e média) para um experimento."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    for name, res in results.items():
        ax1.plot(res['best_dist_history'], label=name)
        ax2.plot(res['avg_dist_history'],  label=name)
    ax1.set_title(f"{title}\nMelhor indivíduo por geração")
    ax2.set_title(f"{title}\nMédia da população por geração")
    for ax in (ax1, ax2):
        ax.set_xlabel("Geração"); ax.set_ylabel(ylabel)
        ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

### Experimento 1 — Operadores de Seleção

Fixamos OX + Swap. Variamos o tamanho do torneio.

In [ ]:
configs_sel = [
    ("Torneio k=3", tournament_k3, ox_crossover, swap_mutation),
    ("Torneio k=7", tournament_k7, ox_crossover, swap_mutation),
]
print("Experimento 1 — Seleção:")
results_sel = run_experiment('eil33', configs_sel)
plot_experiment(results_sel, "Experimento 1 — Seleção (eil33)")

### Experimento 2 — Operadores de Cruzamento

Fixamos Torneio k=3 + Swap. Variamos o cruzamento.

In [ ]:
configs_cross = [
    ("OX",  tournament_k3, ox_crossover,  swap_mutation),
    ("PMX", tournament_k3, pmx_crossover, swap_mutation),
]
print("Experimento 2 — Cruzamento:")
results_cross = run_experiment('eil33', configs_cross)
plot_experiment(results_cross, "Experimento 2 — Cruzamento (eil33)")

### Experimento 3 — Operadores de Mutação

Fixamos Torneio k=3 + OX. Variamos a mutação.

In [ ]:
configs_mut = [
    ("Swap",   tournament_k3, ox_crossover, swap_mutation),
    ("2-opt",  tournament_k3, ox_crossover, two_opt_mutation),
    ("Or-opt", tournament_k3, ox_crossover, or_opt_mutation),
]
print("Experimento 3 — Mutação:")
results_mut = run_experiment('eil33', configs_mut)
plot_experiment(results_mut, "Experimento 3 — Mutação (eil33)")

### Experimento 4 — Melhor Configuração em Todas as Instâncias

Com base nos resultados anteriores, aplicamos a melhor combinação nas três instâncias.

> Ajuste `best_selection`, `best_crossover` e `best_mutation` conforme os resultados dos Experimentos 1–3.

In [ ]:
# ── Ajuste aqui a melhor combinação identificada nos experimentos 1-3 ──
best_selection  = tournament_k3    # ou tournament_k7
best_crossover  = ox_crossover     # ou pmx_crossover
best_mutation   = or_opt_mutation  # ou swap_mutation / two_opt_mutation

configs_best = [("Melhor config.", best_selection, best_crossover, best_mutation)]

print("Experimento 4 — Melhor configuração em todas as instâncias:")
results_all = {}
for inst_name in ['eil33', 'eil51', 'eil76']:
    results_all[inst_name] = run_experiment(inst_name, configs_best)

In [ ]:
for inst_name in ['eil33', 'eil51', 'eil76']:
    plot_experiment(results_all[inst_name],
                    title=f"Experimento 4 — {inst_name}", ylabel="Distância total")

In [ ]:
for inst_name in ['eil33', 'eil51', 'eil76']:
    inst    = instances[inst_name]
    best_res = run_ga(inst['nodes'], inst['node_cap'], inst['cap'], inst['depot'],
                      pop_size=80, n_gen=300, p_cross=0.85, p_mut=0.15,
                      selection_fn=best_selection, crossover_fn=best_crossover,
                      mutation_fn=best_mutation, seed=0, use_pivot=True)
    fit = best_res['best_fitness']
    print(f"{inst_name}: {fit[0]} veículos, distância = {fit[1]:.1f}")
    plot_solution(best_res['best_individual'], inst['nodes'], inst['node_cap'],
                  inst['cap'], inst['depot'], title=f"Melhor solução — {inst_name}")

### Experimento 5 — Impacto da Heurística de Pivôs na Inicialização

Comparamos duas estratégias de inicialização da população:
- `use_pivot=True`: 1 indivíduo gerado pela heurística de pivôs + resto aleatório.
- `use_pivot=False`: toda a população gerada aleatoriamente.

A heurística de pivôs injeta conhecimento geográfico na população inicial. O objetivo é verificar se isso acelera a convergência ou melhora a solução final.

In [ ]:
configs_pivot = [("Melhor config.", best_selection, best_crossover, best_mutation)]

print("Experimento 5 — Com pivô vs. sem pivô:")
results_pivot = {}
for inst_name in ['eil33', 'eil51', 'eil76']:
    print(f"\n  use_pivot=True:")
    r_with    = run_experiment(inst_name, configs_pivot, use_pivot=True)
    print(f"  use_pivot=False:")
    r_without = run_experiment(inst_name, configs_pivot, use_pivot=False)

    # Renomeia para o plot
    results_pivot[inst_name] = {
        "Com pivô"   : r_with["Melhor config."],
        "Sem pivô"   : r_without["Melhor config."],
    }

In [ ]:
for inst_name in ['eil33', 'eil51', 'eil76']:
    plot_experiment(results_pivot[inst_name],
                    title=f"Experimento 5 — Inicialização ({inst_name})",
                    ylabel="Distância total")

## 9. Resultados

*(Preencher após executar os experimentos.)*

### Tabela Resumo — Experimento 4

| Instância | Veículos | Distância Total | Configuração |
|---|---|---|---|
| eil33 | — | — | — |
| eil51 | — | — | — |
| eil76 | — | — | — |

### Análise dos Operadores

- **Seleção:** *(qual torneio performou melhor e por quê)*
- **Cruzamento:** *(comparar OX e PMX)*
- **Mutação:** *(comparar Swap, 2-opt e Or-opt)*

### Impacto da Heurística de Pivôs

*(Descrever se a inicialização com pivôs acelerou a convergência ou melhorou o resultado final.)*

## 10. Conclusões

*(Preencher após a análise dos resultados.)*

O trabalho implementou um Algoritmo Genético completo para o CVRP, com sete operadores genéticos distintos e uma heurística de inicialização baseada em pivôs geográficos. Os principais achados foram:

- ...
- ...

Como trabalho futuro, a heurística de pivôs poderia ser utilizada para gerar toda a população inicial, potencialmente acelerando ainda mais a convergência nas instâncias maiores.